# E07 · 00 语料总览

E07 的第一本 notebook：认识 p0 复制出来的 952 篇 961 观测集语料，为后续的
候选池构造（种子 → 噪声 → 资源/引文扩展 → 代表性案例筛选）摸清底数。

数据来源：`data/processed/e07/corpus/`（p0 从 p4a v1 副本按 e01 n0 观测集筛选复制）。

**口径声明**：layer4 / cite 各层字段是 p4a v1 用当时模型自动抽取的**线索，不是事实**；
描述类字段质量有限，任何进入案例的论断都需要回原文核查。

In [ ]:
import nbio

nbio.banner()
s0 = nbio.summary()

## 语料构成

952 篇全部来自 ACL 2026（n0 观测集定义），按 track 拆开看。

In [ ]:
from collections import Counter

import pandas as pd

paper_ids = sorted(p.name for p in nbio.CORPUS.iterdir() if p.is_dir())
assert len(paper_ids) == s0["n_papers"] == 952

track = Counter(pid.split(".")[1] for pid in paper_ids)
pd.Series(track, name="n_papers").to_frame()

## layer4 资源记录（v1 线索）

每篇论文 `layer4/resource_records.yml` 是 v1 agent 抽取的关联资源清单
（仓库 / 数据集 / benchmark 等）。先把 952 份全部读平，看总量与分布。

In [ ]:
import yaml

rows = []
parse_failures = []
for pid in paper_ids:
    path = nbio.CORPUS / pid / "layer4" / "resource_records.yml"
    try:
        records = yaml.safe_load(path.read_text(encoding="utf-8")) or []
    except yaml.YAMLError:
        parse_failures.append(pid)
        continue
    for item in records:
        rec = item["resource_record"]
        rows.append(
            {
                "paper_id": pid,
                "resource_id": rec["resource_id"],
                "kind": rec["kind"],
                "name": rec["name"],
                "relation_type": rec["paper_relation"]["relation_type"],
                "url": rec["access"].get("url") or "",
                "canonical_url": rec["repository"].get("canonical_url") or "",
                "confidence": rec["provenance"].get("extraction_confidence") or "",
            }
        )

res = pd.DataFrame(rows)
print(f"papers parsed: {len(paper_ids) - len(parse_failures)}/{len(paper_ids)}, "
      f"records: {len(res)}, parse failures: {parse_failures}")
res["paper_id"].value_counts().describe().round(1)

In [ ]:
pd.crosstab(res["kind"], res["relation_type"], margins=True)

### 可建边性：URL 覆盖

p1 要建 paper–resource 边，最硬的锚点是 URL。看两类 URL 字段的覆盖率
（`access.url` 是论文给的入口，`repository.canonical_url` 是 agent 归一化后的仓库地址）。
注意 `kind` 与 URL 语义有重叠（dataset/benchmark 的 access.url 也常指向 GitHub/HuggingFace）。

In [ ]:
cov = res.assign(
    has_access_url=res["url"].str.len() > 0,
    has_canonical=res["canonical_url"].str.len() > 0,
)
cov.groupby("kind")[["has_access_url", "has_canonical"]].agg(["sum", "mean"]).round(2)

In [ ]:
res["confidence"].value_counts()

## 共享资源预览（p1 建边前的朴素观察）

候选池扩展的第一类边是"共享资源"。正式建边要等 p1 讨论归一化规则；
这里只用名字小写做最朴素的聚合，看看 952 篇里哪些资源被多篇论文同时提到，
感受一下扩展边的密度量级。**数字偏乐观**：同名不同物、大小写/别名变体都未处理。

In [ ]:
shared = (
    res.assign(norm=res["name"].str.strip().str.lower())
    .groupby("norm")
    .agg(n_papers=("paper_id", "nunique"), n_records=("paper_id", "size"),
         kinds=("kind", lambda s: "/".join(sorted(set(s)))),
         example=("name", "first"))
    .query("n_papers >= 3")
    .sort_values("n_papers", ascending=False)
)
print(f"被 >=3 篇提到的资源名: {len(shared)} 个")
shared.head(20)

## cite 层：引文锚点

第三类扩展边是引文。`cite/references.jsonl` 是解析出的参考文献条目，
其中带 arxiv id / DOI / URL 的条目才有希望锚定到外部论文身份。先看规模与锚点覆盖。

In [ ]:
import json

ref_stats = []
for pid in paper_ids:
    path = nbio.CORPUS / pid / "cite" / "references.jsonl"
    for line in path.read_text(encoding="utf-8").splitlines():
        obj = json.loads(line)
        refs = obj["references"]
        anchored = sum(1 for r in refs if r["arxiv_ids"] or r["dois"] or r["urls"])
        ref_stats.append({"paper_id": pid, "n_refs": len(refs), "n_anchored": anchored})

refs = pd.DataFrame(ref_stats).set_index("paper_id")
refs["anchored_ratio"] = refs["n_anchored"] / refs["n_refs"].clip(lower=1)
refs.describe().round(2)

## 小结与待讨论

- 语料：952 篇 ACL 2026 论文，mineru / layer4 / cite 三层齐备（唯一已知缺口：
  `2026.acl-long.165` 缺 `agent_judgment.json`，p0 已登记）。
- layer4 资源记录是 p1 建边的主要来源，但字段是 v1 线索，可信度参看上表
  confidence 分布与 URL 覆盖率。
- 共享资源聚合显示扩展边密度可观，p1 需要讨论：canonical_url 归一化规则、
  relation_type 是否参与边语义、引文边如何锚定语料外论文身份。
- 本 notebook 是探索，所有数字不作为文档引用依据；load-bearing 的量将下沉为
  `src/e07/` 脚本产物。

## 消歧后的资源统计（应用 registry）

上面的"共享资源预览"是朴素按名字聚合。现在应用人工策展的消歧注册表
`experiments/e07-p4a-case-pool/registry/resource_disambiguation.yml`（2026-09-12，
kimi 整理 + 26 项 web 核查）重新统计。映射规则按优先级：

1. `noise_keys`（v1 抽取噪声，如 "Mathematical reasoning benchmark"）→ 剔除；
2. `split_within_key`（归一化撞 key 但不同物，如 HumanEval/HumanEval+）→ 按原始拼写拆；
3. `merge_map` + 各条目 `corpus_keys`（同物不同名，如 AIME24/AIME 2024）→ 合并到规范 id；
4. 未注册的 key 默认自成身份（标 `__unregistered__`，多为 singleton）。

**口径声明**：registry 只解决"名字 → 资源身份"，不担保论文与资源的关系描述正确；
`verification` 字段区分 web-verified（有一手来源）/ known（未逐一回源）/ uncertain。

In [ ]:
import re

import yaml

REG_PATH = nbio.REPO_ROOT / "experiments/e07-p4a-case-pool/registry/resource_disambiguation.yml"
reg = yaml.safe_load(REG_PATH.read_text(encoding="utf-8"))

key2id = {}
for entry in reg["canonical_resources"]:
    for k in entry.get("corpus_keys", []):
        key2id[k] = entry["id"]
key2id.update(reg["merge_map"])
split_rules = reg["split_within_key"]
noise_keys = set(reg["noise_keys"])


def norm_name(s):
    return re.sub(r"[^a-z0-9]+", "", s.lower())


def resolve(raw_name):
    """原始名字 -> 规范资源 id；噪声返回 None；未注册返回 __unregistered__<key>。"""
    k = norm_name(raw_name)
    if k in noise_keys:
        return None
    rule = split_rules.get(k)
    if isinstance(rule, dict) and raw_name in rule:
        return rule[raw_name]
    return key2id.get(k, f"__unregistered__{k}")


db = res[res["kind"].isin(["dataset", "benchmark"])].copy()
db["canonical_id"] = db["name"].map(resolve)
n_noise = db["canonical_id"].isna().sum()
print(f"dataset/benchmark 记录 {len(db)} 条：剔除噪声 {n_noise} 条，"
      f"未注册 {db['canonical_id'].astype(str).str.startswith('__unregistered__').sum()} 条")
reg_ids = {r["id"] for r in reg["canonical_resources"]}
print(f"命中 registry 规范条目: {db['canonical_id'].isin(reg_ids).sum()} 条")

In [ ]:
# 未注册的名字里有没有漏网的高频资源？
unreg = db[db["canonical_id"].astype(str).str.startswith("__unregistered__")]
unreg.groupby("canonical_id")["paper_id"].nunique().sort_values(ascending=False).head(10)

### 消歧后的共享资源头部

对比朴素聚合：合并（AIME24+AIME 2024 等）让头部更实，拆分（HumanEval vs +）让身份更准。

In [ ]:
canon = (
    db.dropna(subset=["canonical_id"])
    .groupby("canonical_id")
    .agg(n_papers=("paper_id", "nunique"),
         variants=("name", lambda s: "/".join(sorted(set(s))[:6])))
    .sort_values("n_papers", ascending=False)
)
n_shared = (canon["n_papers"] >= 2).sum()
n_total = len(canon)
print(f"规范资源节点: {n_total} 个（含未注册）；被 >=2 篇共享: {n_shared} 个")
canon.head(25)

In [ ]:
# 同 key 拆分的实际效果：HumanEval 家族
db[db["canonical_id"].isin(["humaneval", "humaneval-plus", "mbpp", "mbpp-plus",
                            "codecontests", "codecontests-plus"])] \
  .groupby(["canonical_id", "name"])["paper_id"].nunique().to_frame("n_papers")

## 消歧后小结

- 共享边密度（>=2 篇的规范资源数）是 p1 建边规模的下界估计；
  未注册高频名若出现，说明 registry 需要补充条目（回到消歧流程）。
- 拆分规则的影响集中在 HumanEval/MBPP 等加号家族，占比小但对案例 E
  （测试充分性）这类依赖确切资源身份的需求是关键路径。
- 本 notebook 的数字仍是探索性统计；进入文档引用前要下沉为脚本产物。